<a href="https://colab.research.google.com/github/emilianomtz0189/pld-monitoreo-transaccional/blob/main/notebooks/01_descarga_y_exploracion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Descarga de datos y exploración

**Proyecto:** sistema de monitoreo transaccional para **Prevención de Lavado de Dinero (PLD)** con el dataset sintético *SAML-D* de Kaggle.

**Objetivo de esta etapa:** conocer el universo transaccional antes de diseñar controles. El **enfoque basado en riesgo** pide calibrar las reglas de monitoreo a la operación real de la institución: cuánto se mueve, por qué canales, hacia qué países y con qué frecuencia aparecen operaciones sospechosas.

**En este notebook:**
1. Preparar el entorno y cargar ~9.5 millones de transacciones con tipos de dato optimizados.
2. Volumen y desbalance: qué tan raras son las operaciones sospechosas.
3. Riesgo por canal (tipo de pago), geografía y moneda.
4. Montos.
5. Tipologías de lavado y verificación de fuga de información.
6. Vista por cuenta y por mes (insumo para las etapas 3 y 4).

> **Regla de oro:** la columna `tipologia` revela la etiqueta. Se usa solo para analizar resultados; **nunca** entra a una regla ni a un modelo.

> **Alcance:** SAML-D es un dataset sintético con montos en libras esterlinas y geografía centrada en Reino Unido. Las referencias a la regulación mexicana son ilustrativas.

## 1. Entorno

En Colab se hacen dos cosas al inicio de cada sesión:
- **Conectar Google Drive**, donde se respalda el dataset procesado y los resultados (los datos no se suben a GitHub).
- **Clonar el repositorio**, para trabajar con la misma estructura de carpetas que en GitHub (`data/`, `src/`, `notebooks/`, `resultados/`).

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "pld-monitoreo-transaccional"
CARPETA_DRIVE = Path("/content/drive/MyDrive/pld-monitoreo-transaccional")  # Respaldo de datos y resultados

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q  # Trae los últimos cambios de GitHub
    os.chdir(ruta_repo)
else:
    CARPETA_DRIVE = None
    if Path.cwd().name == "notebooks":
        os.chdir("..")  # En local, el notebook se ejecuta desde notebooks/

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

### Librerías

Se instalan desde `requirements.txt` para que cualquier sesión use las mismas versiones.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.datos import medir_ahorro_memoria, preparar_datos, resumen_estructura, respaldar_resultados
from src.eda import (
    FORMATO_PERFIL,
    agregar_banderas,
    graficar_por_dia,
    graficar_mensual,
    graficar_mezcla_cuentas,
    graficar_montos,
    graficar_tasa,
    graficar_tipologia_canal,
    guardar_figura,
    guardar_metricas,
    perfil_riesgo,
    resumen_cuentas,
    resumen_desbalance,
    resumen_mensual,
    resumen_montos,
    resumen_tipologias,
    resumen_volumen,
    verificar_fuga_tipologia,
)

config.fijar_semilla()   # SEMILLA = 42 en src/config.py
config.crear_carpetas()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

METRICAS = {}  # Todas las cifras de esta etapa; se guardan en resultados/metricas/

### Registro de versiones

Un sistema de monitoreo debe poder **replicarse**: auditoría interna, el auditor externo y el supervisor deben obtener los mismos resultados. Copia la salida de esta celda en `requirements.txt` para fijar las versiones.

In [ ]:
import importlib.metadata as metadata
import platform

paquetes = ["numpy", "pandas", "pyarrow", "matplotlib", "scikit-learn", "lightgbm", "kagglehub", "psutil"]
print(f"# Probado en Python {platform.python_version()}")
for paquete in paquetes:
    try:
        print(f"{paquete}=={metadata.version(paquete)}")
    except metadata.PackageNotFoundError:
        print(f"# {paquete}: no instalado")

## 2. Datos

### Descarga y carga optimizada
La función `preparar_datos` usa el camino más rápido disponible:
1. Si ya existe el Parquet procesado en Drive, lo copia y lo carga (segundos).
2. Si no, descarga el CSV de Kaggle con `kagglehub`, lo procesa, lo guarda en Parquet y lo respalda en Drive. **La primera vez puede tardar varios minutos.**

Si Kaggle pide credenciales, ver `data/README.md`.

**Por qué importa la memoria:** leer ~9.5 millones de filas como texto puede agotar la RAM de Colab. Por eso cada columna se lee con su tipo:

| Columnas | Tipo | Motivo |
|---|---|---|
| Fecha, hora, monedas, países, tipo de pago, tipología | `category` | Pocos valores distintos repetidos millones de veces: se guardan como código entero + diccionario. |
| Cuentas | `int64` → `int32` si caben | Un `int32` solo llega a 2,147,483,647; la función revisa antes de convertir. |
| Monto | `float32` | Suficiente para guardar libras con centavos; los totales se suman en `float64`. |
| Etiqueta | `int8` | Solo toma 0 o 1. |

Si Colab se queda sin memoria, cambia `POR_PARTES = True`: lee de un millón en un millón (más lento, más seguro).

In [ ]:
POR_PARTES = False

df = preparar_datos(CARPETA_DRIVE, por_partes=POR_PARTES)
memoria_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]} | Memoria: {memoria_mb:,.0f} MB")
df.head()

In [ ]:
# Tipo, memoria, nulos y valores únicos por columna. El diccionario de variables está en data/README.md
resumen_estructura(df)

In [ ]:
# Medición real del ahorro de memoria en las primeras 500 mil filas (solo si el CSV está disponible en esta sesión)
ahorro = medir_ahorro_memoria()
METRICAS["ahorro_memoria"] = ahorro
print(ahorro if ahorro else "CSV no disponible en esta sesión (se cargó desde el Parquet de Drive).")

## 3. Volumen

El volumen define la capacidad que necesita el área de monitoreo: cuántas alertas al día puede revisar un equipo determina qué tan estrictas pueden ser las reglas.

In [ ]:
volumen, por_dia = resumen_volumen(df)
METRICAS["volumen"] = volumen
pd.Series(volumen, name="valor").to_frame()

In [ ]:
fig = graficar_por_dia(por_dia)
guardar_figura(fig, "01_transacciones_por_dia.png")
plt.show()

## 4. Desbalance

Con operaciones sospechosas tan escasas, la **exactitud** (*accuracy*) engaña: un sistema que nunca alerta acertaría casi siempre y no detectaría nada. Por eso el proyecto se evalúa con:
- **Precisión:** de las alertas generadas, qué porcentaje eran realmente sospechosas. Mide la carga de falsos positivos sobre los analistas.
- **Recall (sensibilidad):** de todas las operaciones sospechosas, qué porcentaje se detectó. Mide el riesgo de no identificar algo que debía analizarse y, en su caso, reportarse.

> En una institución real no existe una etiqueta perfecta: lo más cercano son los casos que el área de PLD dictaminó y reportó. En SAML-D, por ser sintético, la etiqueta es completa.

In [ ]:
desbalance = resumen_desbalance(df)
METRICAS["desbalance"] = desbalance
TASA_GLOBAL = df[config.COLUMNA_OBJETIVO].mean()
pd.Series(desbalance, name="valor").to_frame()

## 5. Riesgo por canal: tipo de pago

El efectivo es el canal tradicionalmente más vigilado porque no deja rastro del origen de los fondos; por eso la regulación mexicana le dedica reportes específicos (se detalla en la etapa 2).

La columna **`veces_el_promedio`** compara la tasa sospechosa del canal contra la tasa global: es la forma directa de priorizar controles en un enfoque basado en riesgo.

In [ ]:
por_canal = perfil_riesgo(df, "tipo_pago")
METRICAS["tipo_pago"] = por_canal[["transacciones", "sospechosas", "tasa_sospechosa_pct", "veces_el_promedio"]].round(4).reset_index().to_dict("records")
por_canal.style.format(FORMATO_PERFIL)

In [ ]:
fig = graficar_tasa(por_canal, "Tasa de transacciones sospechosas por tipo de pago", TASA_GLOBAL)
guardar_figura(fig, "02_tasa_por_tipo_pago.png")
plt.show()

## 6. Riesgo geográfico y de moneda

Las operaciones transfronterizas, y en especial las que involucran **jurisdicciones de alto riesgo o no cooperantes** (listas del GAFI), son un factor de riesgo clásico: dificultan rastrear origen y destino de los recursos.

- **Internacional:** banco de origen y de destino en distinto país.
- **Cambio de moneda:** moneda pagada distinta de la recibida.

In [ ]:
df = agregar_banderas(df)

for columna in ["es_internacional", "cambio_moneda"]:
    tabla = perfil_riesgo(df, columna)
    METRICAS[columna] = tabla[["transacciones", "pct_transacciones", "tasa_sospechosa_pct", "veces_el_promedio"]].round(4).reset_index().to_dict("records")
    print(f"\n{columna} (1 = sí)")
    display(tabla[["transacciones", "pct_transacciones", "sospechosas", "tasa_sospechosa_pct", "veces_el_promedio"]].style.format(FORMATO_PERFIL))

In [ ]:
# País del banco que RECIBE (a dónde va el dinero). Se excluyen países con menos de 1,000 transacciones: su tasa sería poco confiable.
MINIMO_TRANSACCIONES = 1_000
por_pais_destino = perfil_riesgo(df, "pais_banco_destino", minimo=MINIMO_TRANSACCIONES)
METRICAS["pais_banco_destino"] = por_pais_destino[["transacciones", "tasa_sospechosa_pct", "veces_el_promedio"]].round(4).reset_index().to_dict("records")
por_pais_destino.style.format(FORMATO_PERFIL)

In [ ]:
fig = graficar_tasa(por_pais_destino, "Tasa sospechosa por país del banco destino", TASA_GLOBAL)
guardar_figura(fig, "03_tasa_por_pais_destino.png")
plt.show()

In [ ]:
# País del banco que ENVÍA (de dónde viene el dinero)
por_pais_origen = perfil_riesgo(df, "pais_banco_origen", minimo=MINIMO_TRANSACCIONES)
METRICAS["pais_banco_origen"] = por_pais_origen[["transacciones", "tasa_sospechosa_pct", "veces_el_promedio"]].round(4).reset_index().to_dict("records")
por_pais_origen.style.format(FORMATO_PERFIL)

## 7. Montos

La intuición dice "lavado = montos grandes", pero varias tipologías hacen lo contrario: **fraccionan** en montos menores para no rebasar umbrales de reporte. Si los montos sospechosos se parecen a los normales, una regla basada solo en monto alto será insuficiente.

In [ ]:
montos = resumen_montos(df)
METRICAS["montos"] = montos.round(2).to_dict()
montos.style.format("{:,.2f}")

In [ ]:
fig = graficar_montos(df)
guardar_figura(fig, "04_distribucion_montos.png")
plt.show()

In [ ]:
# Monto mediano por tipo de pago: normal (0) vs. sospechosa (1)
df.pivot_table(index="tipo_pago", columns=config.COLUMNA_OBJETIVO, values="monto",
               aggfunc="median", observed=True).round(2)

## 8. Tipologías (solo análisis)

Una **tipología** es un patrón conocido de lavado: fraccionar depósitos, concentrar fondos de muchas cuentas en una, mover recursos en cadena, entre otros. Las autoridades publican tipologías para orientar a las instituciones sobre qué buscar.

### Verificación de fuga de información
Si cada tipología es 100 % normal o 100 % sospechosa, conocer la tipología equivale a conocer la respuesta. Esta verificación justifica excluirla de reglas y modelos.

In [ ]:
fuga = verificar_fuga_tipologia(df)
es_pura = fuga["pct_sospechosa"].isin([0, 100]).all()
METRICAS["tipologias"] = {
    "total": int(len(fuga)),
    "normales": int((fuga["pct_sospechosa"] == 0).sum()),
    "sospechosas": int((fuga["pct_sospechosa"] == 100).sum()),
    "revela_la_etiqueta": bool(es_pura),
}
print(METRICAS["tipologias"])
fuga.style.format({"transacciones": "{:,.0f}", "pct_sospechosa": "{:.1f}%"})

### Perfil de las tipologías sospechosas
Este perfil sirve en la etapa 2 para medir qué tipologías detecta cada regla.

In [ ]:
tipologias = resumen_tipologias(df)
METRICAS["tipologias"]["detalle"] = tipologias[["transacciones", "pct_de_las_sospechosas", "canal_principal", "pct_internacional"]].round(2).reset_index().to_dict("records")
tipologias.style.format({"transacciones": "{:,.0f}", "monto_mediano": "£{:,.2f}", "cuentas_que_envian": "{:,.0f}",
                         "pct_de_las_sospechosas": "{:.2f}%", "pct_internacional": "{:.1f}%"})

In [ ]:
fig = graficar_tipologia_canal(df, orden=list(tipologias.index))
guardar_figura(fig, "05_tipologia_por_canal.png")
plt.show()

## 9. Vista por cuenta

En la práctica, el analista no revisa transacciones aisladas sino **casos por cliente**: abre el expediente y revisa todo su historial. Por eso importa cuántas cuentas están involucradas y si las cuentas con operaciones sospechosas también realizan operaciones normales que las camuflan.

In [ ]:
cuentas, fraccion = resumen_cuentas(df)
METRICAS["cuentas"] = cuentas
pd.Series(cuentas, name="valor").to_frame()

In [ ]:
fig = graficar_mezcla_cuentas(fraccion)
guardar_figura(fig, "06_mezcla_en_cuentas.png")
plt.show()

## 10. Vista por mes

El modelo de la etapa 4 se entrena con meses **anteriores** y se evalúa con meses **posteriores**, como ocurriría en producción. Esta tabla muestra qué meses están completos y si la tasa sospechosa es estable en el tiempo.

In [ ]:
mensual = resumen_mensual(df)
METRICAS["mensual"] = [{"mes": str(m), **{k: (round(float(v), 4) if k == "tasa_sospechosa_pct" else int(v)) for k, v in fila.items()}}
                       for m, fila in mensual.iterrows()]
mensual.style.format({"transacciones": "{:,.0f}", "sospechosas": "{:,.0f}", "tasa_sospechosa_pct": "{:.4f}%"})

In [ ]:
fig = graficar_mensual(mensual, TASA_GLOBAL)
guardar_figura(fig, "07_tasa_mensual.png")
plt.show()

## 11. Resumen y guardado

Todas las cifras de este notebook se guardan en `resultados/metricas/01_exploracion.json`, que es la única fuente de cifras para el README. También se respaldan en Drive.

In [ ]:
guardar_metricas(METRICAS, "01_exploracion.json")
if CARPETA_DRIVE is not None:
    respaldar_resultados(CARPETA_DRIVE)

v, d, c = METRICAS["volumen"], METRICAS["desbalance"], METRICAS["cuentas"]
canal = por_canal.index[0]
print(f'''
RESUMEN ETAPA 1
- {v["transacciones"]:,} transacciones de {v["cuentas_unicas"]:,} cuentas, del {v["fecha_inicial"][:10]} al {v["fecha_final"][:10]} ({v["dias_con_operacion"]} días).
- Promedio de {v["transacciones_por_dia_promedio"]:,} transacciones por día.
- {d["transacciones_sospechosas"]:,} sospechosas = {d["tasa_sospechosa_pct"]}% (una de cada {d["una_de_cada"]:,}).
- Un sistema que nunca alerta tendría {d["exactitud_si_nada_es_sospechoso_pct"]}% de exactitud.
- Canal con mayor tasa sospechosa: {canal} ({por_canal.loc[canal, "veces_el_promedio"]:.2f} veces el promedio).
- {METRICAS["tipologias"]["total"]} tipologías ({METRICAS["tipologias"]["sospechosas"]} sospechosas); ¿revelan la etiqueta?: {METRICAS["tipologias"]["revela_la_etiqueta"]}.
- {c["cuentas_con_alguna_operacion_sospechosa"]:,} cuentas con alguna operación sospechosa ({c["pct_del_total_de_cuentas"]}% del total).
''')

**Siguiente paso:** notebook `02_reglas_monitoreo.ipynb`: reglas de monitoreo por tipología y su medición con precisión, recall y alertas por día.

> Los archivos de `data/` no se suben a GitHub. En una sesión nueva de Colab basta con volver a ejecutar este notebook: el Parquet respaldado en Drive se carga en segundos.